# Non-Uniform Space-Filling Design

This notebook builds a 20-run non-uniform
space-filling design over a 2-D input space, where a weight column emphasizes
some regions more than others. The **maximum weight ratio (MWR)** controls how
strongly the higher-weight regions are concentrated (MWR = 1 is a uniform
design; larger MWR packs more points into high-weight regions).

## 1. Load and inspect the candidate set with weights

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from idaes_sdoe import ColumnRoles, load_csv, prepare_design_setup
from idaes_sdoe.design import design_nonuniform
from idaes_sdoe.plotting import plot_nonuniform_weights, plot_pair_matrix
from idaes_sdoe.scaling import scale_weights

candidate = load_csv(Path("supporting_data/NUSFex1.csv"))
print(f"{len(candidate)} candidate points")
candidate.describe().loc[["min", "max"]]

The raw weight ``RawWT`` shown in color over the input grid (highest toward the upper-left corner):

In [ ]:
go.Figure(
    go.Scatter(x=candidate["X1"], y=candidate["X2"], mode="markers",
               marker={"color": candidate["RawWT"], "colorscale": "Viridis",
                       "colorbar": {"title": "RawWT"}, "size": 9})
).update_layout(title="Candidate weights over the input grid",
                xaxis_title="X1", yaxis_title="X2", template="plotly_white",
                width=620, height=520)

The same candidate set with the weight shown as point size:

In [ ]:
w = candidate["RawWT"]
scaled = (w - w.min()) / (w.max() - w.min())
go.Figure(
    go.Scatter(x=candidate["X1"], y=candidate["X2"], mode="markers",
               marker={"size": 4 + 16 * scaled, "color": "rgba(31,119,180,0.6)"})
).update_layout(title="Candidate weights shown as point size",
                xaxis_title="X1", yaxis_title="X2", template="plotly_white",
                width=620, height=520)

## 2. Construct designs for several MWR values

Non-uniform space filling uses the maximin criterion. We use **Direct** scaling
(a linear map of the weights into ``[1, MWR]``) and build 20-run designs for
MWR values of 5, 10, and 30. Criterion values are only
comparable across designs of the same size *and* the same MWR.

In [ ]:
NUM_RESTARTS = 30  # number of random starts

setup = prepare_design_setup(
    candidate=candidate, roles=ColumnRoles(inputs=["X1", "X2"], weight="RawWT")
)
results = {}
for mwr in [5, 10, 30]:
    results[mwr] = design_nonuniform(
        setup=setup, design_size=20, num_restarts=NUM_RESTARTS, mwr=mwr,
        scale_method="direct_mwr", random_state=42,
    )
    print(f"MWR={mwr:2d}: criterion={results[mwr].criterion_value:.4f}")

## 3. Closest-Distance-by-Weight (CDBW) and pairwise plots

The CDBW plot has two parts: the lower histogram shows the candidate weights
scaled into ``[1, MWR]``, and the upper panel draws one vertical line per design
point at its (scaled) weight — showing how much emphasis the design placed on
high-weight points. As MWR grows, the selected points shift toward the maximum
weight.

In [ ]:
def cdbw(mwr):
    result = results[mwr]
    return plot_nonuniform_weights(
        result.scaled_design[["X1", "X2"]].to_numpy(),
        result.scaled_design["RawWT"].to_numpy(),
        scale_weights(candidate["RawWT"].to_numpy(), method="direct_mwr", mwr=mwr),
        title=f"Closest distance by weight (MWR = {mwr})",
    )

cdbw(5)

In [ ]:
cdbw(10)

In [ ]:
cdbw(30)

Pairwise scatter of the design points over the candidate set, for the least and most concentrated MWR values:

In [ ]:
plot_pair_matrix(results[5].design, columns=["X1", "X2"], candidate=candidate,
                 title="NUSF design (MWR = 5)")

In [ ]:
plot_pair_matrix(results[30].design, columns=["X1", "X2"], candidate=candidate,
                 title="NUSF design (MWR = 30)")

Increasing the MWR moves more design points into the higher-weight region while
still filling the rest of the space — letting the experimenter tune the density
of points to match where information is most valuable.